In [ ]:
!pip install -U transformers cedarpy datasets peft accelerate bitsandbytes trl

# Cedar Policy Generation with LLMs

## Introduction
Access controls is an important part of modern day software and technology. Application teams and developers do not want uncontrolled access to protect consumer's data since there is a huge amount of data on the internet.

## What is Cedar?
According to a paper published by Amazon Web Services (AWS), Cedar is an authorization policy language designed to be fast, safe, and analyzable. Using Cedar allows applications to separate their business logic from their access decision logic. Cedar has different kinds of policies that can be defined in 4 major categories: Role-based, Attribute-based, Relationship-based, and Context-based.

## Challenge of using Cedar
The challenge of application teams to use Cedar for their application is learning and understanding how to convert their use cases into this new language called Cedar. This causes delays in the development of application, which becomes a big problem in a market of evolving technologies being developed. The challenges developers face is
1. Understanding what each of the key words mean (principal, resource, and action)
2. The capabilities that Cedar policies have (having broad policies to specific policies)

## What I want to solve
I want to see if these access control policies can be generated given an application team's use cases using an LLM model. Would it be possible to just be sufficient with prompting an LLM? Would fine-tuning an LLM for this unique use-case be better? How can I further tune it to grow with new access control policies that may be needed as technology grows?


# Methodlogy
1. Few-Shot prompting
    - The key idea here is to have a baseline on the accuracy of LLMs that currently exist by giving it a couple prompts to produce the proper cedar policy.
2. Fine tuning with unstructured data
    - The idea here is to see the performance increase with the fine tuned of an LLM 
3. Fine tuning with structured data
    - This should be the best performing model with clear structured data that LLMs understand and can use

In [ ]:
# Common imports and configuration
import json
import re
from cedarpy import validate_policies, ValidationResult
from transformers import AutoTokenizer, AutoModelForCausalLM

# model name to be used for this project
model_name = "meta-llama/Llama-3.1-8B-Instruct"

# System prompt that is used across all methods
SYSTEM_PROMPT = """
You are an expert in Cedar policy language.

Convert a list natural language access control requirements into:
1. A SINGLE unified valid Cedar schema
2. A SET of valid Cedar Policies (one per use case)

RULES:
Schema Rules:
- Generate ONE shared schema for ALL use cases
- Do not duplicate entities, actions, and attributes
- Keep schema minimal but sufficient
- Reuse entities/actions across use cases when possible

Policy Rules:
- Generate ONE policy per use case
- Each policy must be valid in respect to the schema
- Use consistent naming that is defined in the schema

General:
- Generate the namespace as CHANGEME_APP_NAME
- Use correct Cedar syntax
- Do NOT escape quotes
- Output ONLY Cedar code (no explanations)
- Do NOT include explanations, markdown, or commentary
- You MUST output BOTH schema and policy
- If required information is missing, output a single Cedar comment explaining what is missing, and nothing else.
- Use this EXACT format for the output:

Schema:
...schema here...

Policy:
- POLICY_0:
  ...policy here...
- POLICY_1:
  ...policy here...
...rest of policies...
""".strip()

unstructured_data = []
with open("unstructured_dataset.jsonl", "r") as f:
  for line in f:
    data_line = json.loads(line)
    prompts = [
      {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": data_line["input"].strip()},
        {"role": "assistant", "content": data_line["output"].strip()}
    ]
    unstructured_data.append(prompts)
    
def validate_policy_file(policy_content: str, schema_content: str):
    """Validate a Cedar Policy, return if validation errors

    Args:
        policy_content (str): content of the cedar policy
        schema_content (str): content of the cedar schema
    """
    validation_result: ValidationResult = validate_policies(policy_content, schema_content)
    if not validation_result.validation_passed:
        print("Policy is invalid. Errors:")
        for error in validation_result.errors:
            print(f"- {error}")
        return False
    return True

# 1. Few-Shot Prompting
Here I will be using a few prompts to show give the LLM context on what the input is and the expected output. Then I will use Cedarpy, an open-source cedar policy evaluate written in python, to validate the outputted cedar schema and policies. 

In [ ]:
FEW_SHOT_EXAMPLES = unstructured_data[:6]
FEW_SHOT_EXAMPLES.append(
    {
        "role": "user", 
        "content": "Ecommerce system requirements: Customers must be able to browse and purchase products. Sellers must be able to manage their listings and view associated orders. Administrative users must have the ability to remove any product listing."
    }

)

few_shot_prompt_tokenizer = AutoTokenizer.from_pretrained(model_name)
few_shot_prompt_model = AutoModelForCausalLM.from_pretrained(model_name)

few_shot_input_prompts = few_shot_prompt_tokenizer.apply_chat_template(
    FEW_SHOT_EXAMPLES, 
    add_generation_prompt=True,
    return_tensors="pt",
  ).to(few_shot_prompt_model.device)

few_shot_output_encoded = few_shot_prompt_model.generate(**few_shot_input_prompts, max_new_tokens=2048)
few_shot_output = few_shot_prompt_tokenizer.decode(few_shot_output_encoded, skip_special_tokens=True)
print(f"FEW SHOT OUTPUT:\n{few_shot_output}")

few_shot_policies = re.findall(r"- (POLICY_\d+)\s*(.*?)\s*(?=- POLICY_\d+|$)", few_shot_output, re.DOTALL)

few_shot_schema = re.search(r"Schema:\s*(.*?)\s*Policy:", few_shot_output, re.DOTALL).group(1).strip()
print(f"Validating with Schema:\n{few_shot_schema}")
for policy_name, policy_body in few_shot_policies:
  if validate_policy_file(policy_body.strip(), few_shot_schema):
    print(f"{policy_name} is valid.")
  else:
    print(f"{policy_name} is invalid.")

# 2. Fine tuning with unstructured data

In [ ]:
import torch
from datasets import Dataset
from transformers import BitsAndBytesConfig, SFTTrainer, SFTConfig, LoraConfig

bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.float32
)

unstructured_data_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=bnb_config)
unstructured_data_tokenizer = AutoTokenizer.from_pretrained(model_name)

training_prompts = []
for data in unstructured_data:
    training_prompts.append({"text": unstructured_data_tokenizer.apply_chat_template(data, tokenize=False)})
    
dataset = Dataset.from_list(training_prompts)
unstructured_training_set = dataset.train_test_split(test_size=0.1, seed=42)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
sft_config = SFTConfig(
    gradient_checkpointing=True,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=2,
    num_train_epochs=10,
    learning_rate=3e-4,
    optim="paged_adamw_8bit",
    logging_steps=10,
    dataset_text_field="text",
    output_dir="model/out/cedar-model-unstructured",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=2
)

unstructured_data_trainer = SFTTrainer(
    model=unstructured_data_model,
    processing_class=unstructured_data_tokenizer,
    train_dataset=unstructured_training_set["train"],
    eval_dataset=unstructured_training_set["test"],
    peft_config=lora_config,
    args=sft_config,
)

unstructured_data_trainer.train()
unstructured_data_trainer.save_model("model/out/cedar-model-unstructured")
unstructured_data_tokenizer.save_pretrained("model/out/cedar-model-unstructured")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[
  {
    "role": "system",
    "content": "You are an expert in Cedar policy language.\nConvert natural language access control requirements into valid Cedar policies.\n\nRules:\n- Generate the namespace as CHANGEME_APP_NAME\n- Use correct Cedar syntax\n- Do NOT escape quotes\n- Output ONLY valid Cedar policy code\n- Do NOT include explanations, markdown, or commentary\n- If required information is missing, output a single Cedar comment explaining what is missing, and nothing else."
  },
  {
    "role": "user",
    "content": "Allow an Admin to do any action on any resource."
  },
  {
    "role": "assistant",
    "content": "permit (\n            principal in CHANGEME_APP_NAME::Role::\"Admin\",\n            action,\n            resource\n        );"
  },
  {
    "role": "user",
    "content": "Allow nurses to view any patient record."
  },
  {
    "role": "assistant",
    "content": "permit (\n            principal is CHANGEME_APP_NAME::Nurse,\n            action == CHANGEME_APP_NAME:

In [ ]:


bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.float32
)

structured_data_model = AutoTokenizer.from_pretrained(model_name)
structured_data_tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto", quantization_config=bnb_config)

structured_data = []
with open("structured_dataset.jsonl", "r") as f:
  for line in f:
    data_line = json.loads(line)
    prompts = [
      {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": data_line["input"].strip()},
        {"role": "assistant", "content": data_line["output"].strip()}
    ]
    structured_data.append({"text": structured_data_tokenizer.apply_chat_template(prompts, tokenize=False)})
    
structured_dataset = Dataset.from_list(structured_data)
structured_training_set = structured_dataset.train_test_split(test_size=0.1, seed=42)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
sft_config = SFTConfig(
    gradient_checkpointing=True,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=2,
    num_train_epochs=10,
    learning_rate=3e-4,
    optim="paged_adamw_8bit",
    logging_steps=10,
    dataset_text_field="text",
    output_dir="model/out/cedar-model-structured",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=2
)

unstructured_data_trainer = SFTTrainer(
    model=unstructured_data_model,
    processing_class=unstructured_data_tokenizer,
    train_dataset=structured_training_set["train"],
    eval_dataset=structured_training_set["test"],
    peft_config=lora_config,
    args=sft_config,
)

unstructured_data_trainer.train()
unstructured_data_trainer.save_model("model/out/cedar-model-structured")
unstructured_data_tokenizer.save_pretrained("model/out/cedar-model-structured")

Successfully validated policy:
permit (
          principal is CHANGEME_APP_NAME::User,
          action == CHANGEME_APP_NAME::Action::"ViewPost",
          resource is CHANGEME_APP_NAME::Post
        )
        when {
          resource has owner &&
          resource.owner has followers &&
          principal in resource.owner.followers
        };


From the above result, we can see that the LLM was unable to generate a complex policy with a stripped down schema that I created to be compatible with the policy. This could be due to the result of Human Error or Computer error. Let's take a look at getting the LLM to generate both.

## Creating Schema + 1 policy

This result is much better where you can see the clear definitions of the actions and resources and the removal of manual schema creation. However, this does not fit the case for multiple policies.

# Fine tuning the LLM

In [27]:
ft_tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "auto",
    torch_dtype=torch.float16,
    load_in_4bit=true
)



NameError: name 'torch' is not defined